In [ ]:
import sys
import os
import time as _time
import pandas as pd
import inspect
from pathlib import Path


 
# Find the project root
def find_project_root(marker="src/data_prep.py"):

    # Calls the class working directory from pathlib, and captures the starting location
    here = Path.cwd()

    # Creates an upward search path, checking one folder layer at a time until
    # candidate (the project root) is returned 
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
        
    # Incase the file is not found
    raise FileNotFoundError("Could not find the file")

# Establishing the project paths and root
project_root = find_project_root()
src_path = project_root / "src"
data_path = project_root / "data"



print(f"Project root confirmed at: {project_root}")

# Insert src directory at top priority
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import the processing function
from data_prep import build_full_dataset
from data_prep import fetch_sequence
from data_prep import build_ClinVar_dataset 

#Sanity checks to see if the function is actually returning the DNA sequence
print(fetch_sequence("14", 23412740))       


ClinVar_path = str(data_path / "raw" / "variant_summary.txt.gz")

#start = _time.time()
#test_df = build_ClinVar_dataset(ClinVar_path, "test_out.csv", limit=500)
#elapsed = _time.time() - start

#print(f"{elapsed:.1f}s for {len(test_df)} rows kept")

# Defining the ClinVar file path (includes lots of variants, which are filtered in data_prep)


# Unlike ClinVar, gnomAD includes seperated data for each gene. This takes
# the paths for each of the three genes.
gnomAD_csv_paths = {
    "MYH7": str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN": str(data_path / "raw" / "gnomAD_TTN.csv")
}
out_path = str(data_path / "processed" / "dataset.csv")

# Runs data preparation pipeline and calls build_ClinVar_dataset from data_prep
# note that build_full_dataset was originally ran, but for the second time running, because
# I already loaded all the benign gnomAD variants, only build_ClinVar_dataset is neccesary
print("Starting data preparation pipeline")

final_df = build_ClinVar_dataset(
    ClinVar_path=ClinVar_path,
    out_path=out_path,
    old_frac=0.5,   # this part was already saved
    new_frac=1.0,   # new parts that need to be added
)

# Configure Pandas display options and show results
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)





print("\n--- ENTIRE FINISHED DATASET ---")
display(final_df)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from train import cap_benign 
import annotate

data_path = project_root/"data"
dataset_path = data_path/"processed"/"dataset.csv"
# Reads the dataset_path, same seed for reproducibility
df = pd.read_csv(dataset_path)
print(dataset_path)
seed = 42

# Calls cap_benign from train to cap the benign variants amount
capped = cap_benign(df, max_benign=None, seed=seed)

# Makes the datasets also inside of train, uses data stratification
rest_df, demo_df = train_test_split(capped, test_size= 0.02, stratify=capped["label"], random_state=seed)
train_df, test_df = train_test_split(rest_df, test_size=0.10/0.98, stratify=rest_df["label"], random_state=seed)

# Prints the value count of the amount of rows and each label
print(f"test_df = {len(test_df)} rows")
print(test_df["label"].value_counts())

# Calls build_annotation_subset from annotate.py
subset = annotate.build_annotation_subset(df, test_df)

# Calls annotate_dataset from annotate.py, with no in_path, and a csv output. 
annotated = annotate.annotate_dataset (None, str(data_path / "processed" / "dataset_annotated.csv"), df=subset)

results = annotate.revel_cadd_benchmark(annotated, test_df)

# Returns the results by using .items() to return a (key, value) pair each time
print("\n--- Binary Pathogenic-vs-Benign AUC-ROC, same 716 test variants ---")
for name, auc in results.items():
    print(f"{name:35s} {auc:.3f}")


ModuleNotFoundError: No module named 'train'

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


import diagnose
dataset_path = data_path / "processed" / "dataset.csv"
diagnose.run(dataset_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- CONSEQUENCE vs LABEL ---
label        Benign  DCM  HCM
consequence                  
missense        625    0  234
noncoding      2789    0  111
nonsense          1  974  112
synonymous     7466    0    0
unknown         240    0    0

--- GENE vs LABEL ---
label   Benign  DCM  HCM
gene                    
MYBPC3     692    0  234
MYH7      1050    0  223
TTN       9379  974    0
Benign capped: 7155
Genetic-code rule (path vs benign) : 0.931
Stop-codon rule   (DCM vs rest)    : 0.989
Gene-name rule    (HCM vs DCM)     : 1.000


In [ ]:

import sys
from pathlib import Path
import pandas as pd

def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project root :", project_root)
print("src on path  :", str(src_path) in sys.path)

%load_ext autoreload
%autoreload 2

clinvar_dataset_path = str(data_path / "processed" / "dataset.csv")
clinvar_raw_path     = str(data_path / "raw" / "variant_summary.txt.gz")



In [ ]:
# Makes sure not to use old versions of the same function, so that changing annotate later on works without having to push the code.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Iterative project root finding
def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")

project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from consequence import parse_consequence

# Prints the dataset shape
dataset_path = data_path / "processed" / "dataset.csv"
df = pd.read_csv(dataset_path)
print("dataset shape:", df.shape)

# Runs parse_consequence once on every value in the name column and returns a new column
df["consequence"] = df["name"].apply(parse_consequence)

# Crosstab counts how many rows have each combination of the two columns
# Normalize="columns" turns each column into fractions of that column's total
counts = pd.crosstab(df["consequence"], df["label"])
frac   = pd.crosstab(df["consequence"], df["label"], normalize="columns")

print("\n=== COUNTS ===")
print(counts)
print("\n=== FRACTION OF EACH CLASS ===")
print(frac.round(3))

# This is so that a missing row or column returns 0 instead of raising a KeyError.
def pct(cons, label):
    if cons in frac.index and label in frac.columns:
        return frac.loc[cons, label] * 100
    return 0.0



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
dataset shape: (12552, 8)
label               Benign  DCM  HCM
gene   consequence                  
MYBPC3 missense         22    0   23
       noncoding       304    0  102
       nonsense          0    0  109
       synonymous      358    0    0
       unknown           8    0    0
MYH7   missense          7    0  211
       noncoding       404    0    9
       nonsense          0    0    3
       synonymous      633    0    0
       unknown           6    0    0
TTN    missense        596    0    0
       noncoding      2081    0    0
       nonsense          1  974    0
       synonymous     6475    0    0
       unknown         226    0    0


In [ ]:

import sys
import os
import time
from pathlib import Path
import pandas as pd


def find_project_root(marker="src/data_prep.py"):
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError("Could not find the project root")


project_root = find_project_root()
src_path  = project_root / "src"
data_path = project_root / "data"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project root :", project_root)
print("src on path  :", str(src_path) in sys.path)

%load_ext autoreload
%autoreload 2

clinvar_dataset_path = str(data_path / "processed" / "dataset.csv")        
clinvar_raw_path     = str(data_path / "raw" / "variant_summary.txt.gz")   
out_path             = str(data_path / "processed" / "dataset_full.csv")   
smoke_path           = str(data_path / "processed" / "SMOKE_TEST.csv")     

gnomAD_csv_paths = {
    "MYH7":   str(data_path / "raw" / "gnomAD_MYH7.csv"),
    "MYBPC3": str(data_path / "raw" / "gnomAD_MYBPC3.csv"),
    "TTN":    str(data_path / "raw" / "gnomAD_TTN.csv"),
}

for name, p in [("clinvar_dataset", clinvar_dataset_path),
                ("clinvar_raw", clinvar_raw_path),
                *gnomAD_csv_paths.items()]:
    print(f"{name:16s} exists: {Path(p).exists()}")

assert out_path != clinvar_dataset_path, "out_path would overwrite the 18-hour build"
assert smoke_path != clinvar_dataset_path, "smoke_path would overwrite the 18-hour build"
print("\noutput paths are safe")

from data_prep import consequence_targets
from consequence import parse_consequence

df = pd.read_csv(clinvar_dataset_path)
print("rows:", len(df))
print("columns:", list(df.columns), "\n")

if "consequence" not in df.columns:
    df["consequence"] = df["name"].apply(parse_consequence)

print(pd.crosstab([df["gene"], df["consequence"]], df["label"]).to_string(), "\n")

total = 0
for g in ["MYH7", "MYBPC3", "TTN"]:
    t = consequence_targets(df, g, ratio=1.0)
    total += sum(t.values())
    print(f"{g:7s} {t}")
print(f"\ntotal controls requested: {total}")

# EXPECTED:
#   MYH7    {'missense': 204, 'nonsense': 3}
#   MYBPC3  {'missense': 1, 'nonsense': 109}
#   TTN     {'nonsense': 973}

from data_prep import build_full_dataset

smoke = build_full_dataset(
    clinvar_dataset_path = clinvar_dataset_path,
    clinvar_raw_path     = clinvar_raw_path,
    gnomAD_csv_paths     = gnomAD_csv_paths,
    out_path             = smoke_path,
    faf_threshold        = 0.0,
    ratio                = 1.0,
    limit                = 20,
)

t0 = time.time()

full = build_full_dataset(
    clinvar_dataset_path = clinvar_dataset_path,
    clinvar_raw_path     = clinvar_raw_path,
    gnomAD_csv_paths     = gnomAD_csv_paths,
    out_path             = out_path,
    faf_threshold        = 0.0,
    ratio                = 1.0,
)

print(f"\nfinished in {(time.time() - t0) / 60:.1f} minutes")

check = pd.read_csv(out_path)

print("rows:", len(check))
print(check["source"].value_counts().to_string(), "\n")

print("ref_sequence on every row :", bool(check["ref_sequence"].notna().all()))
print("mutant windows all 201    :", bool((check["sequence"].str.len() == 201).all()))
print("ref windows all 201       :", bool((check["ref_sequence"].str.len() == 201).all()))
print("no duplicate variants     :",
      not check.duplicated(subset=["chrom", "pos", "ref", "alt"]).any(), "\n")

print(pd.crosstab([check["gene"], check["consequence"]], check["label"]).to_string())



project root : c:\Users\jaira\Desktop\Official cardiac project\Cardiac-project
src on path  : True
clinvar_dataset  exists: True
clinvar_raw      exists: True
MYH7             exists: True
MYBPC3           exists: True
TTN              exists: True

output paths are safe
rows: 12552
columns: ['sequence', 'label', 'gene', 'pos', 'chrom', 'ref', 'alt', 'name'] 

label               Benign  DCM  HCM
gene   consequence                  
MYBPC3 missense         22    0   23
       noncoding       304    0  102
       nonsense          0    0  109
       synonymous      358    0    0
       unknown           8    0    0
MYH7   missense          7    0  211
       noncoding       404    0    9
       nonsense          0    0    3
       synonymous      633    0    0
       unknown           6    0    0
TTN    missense        596    0    0
       noncoding      2081    0    0
       nonsense          1  974    0
       synonymous     6475    0    0
       unknown         226    0    0 

MYH7  

ValueError: values cannot be used without an aggfunc.

In [ ]:
from model import CardiacCNN
import torch

# Build a fresh, untrained model with the same architecture
loaded_model = CardiacCNN(seq_len=201, n_classes=3)

# Load the saved weights from Drive into it
loaded_model.load_state_dict(torch.load("/content/drive/MyDrive/model_state_backup_0.954.pt"))
loaded_model.eval()

print(loaded_model)